In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Assignment").getOrCreate()

In [2]:
df = spark.read.csv("Lung Cancer.csv", header=True)

Task 1: Write a function that removes duplicate rows, ensures correct data types for numerical and date columns and converts all ‘yes’/ ‘no’ type fields into 1/0 format.  

In [3]:
from pyspark.sql.functions import col, when, lower, trim, to_date

df = df.dropDuplicates()

df = df.withColumn(
    "family_history",
    when(lower(trim(col("family_history"))) == "yes", 1)
    .when(lower(trim(col("family_history"))) == "no", 0)
    .otherwise(None)
)

df = df.withColumn("age", col("age").cast("double")) \
       .withColumn("bmi", col("bmi").cast("double")) \
       .withColumn("survived", col("survived").cast("int")) \
       .withColumn("hypertension", col("hypertension").cast("int")) \
       .withColumn("diagnosis_date", to_date("diagnosis_date")) \
       .withColumn("end_treatment_date", to_date("end_treatment_date"))

In [4]:
df.show(5, truncate=False)
df.printSchema()

+----+----+------+--------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|id  |age |gender|country |diagnosis_date|cancer_stage|family_history|smoking_status|bmi |cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+----+----+------+--------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|441 |55.0|Female|Austria |2015-06-07    |Stage I     |1             |Current Smoker|31.7|245              |1           |0     |0        |0           |Combined      |2017-01-30        |1       |
|552 |49.0|Female|Portugal|2019-01-02    |Stage II    |0             |Former Smoker |25.3|207              |0           |0     |0        |0           |Combined      |2020-05-29        |0       |
|1046|51.0|Male  |Germany

Task 2: Write a function that adds a new column, treatment_duration_days, which calculates the number of days between the diagnosis and the end of treatment. Then, return the average treatment duration for each treatment type.  

In [5]:
from pyspark.sql.functions import datediff, avg

df2 = df.withColumn(
    "treatment_duration_days",
    datediff(col("end_treatment_date"), col("diagnosis_date"))
)

df2.groupBy("treatment_type") \
   .agg(avg("treatment_duration_days").alias("avg_days")) \
   .show()

+--------------+------------------+
|treatment_type|          avg_days|
+--------------+------------------+
|     Radiation|458.40320462900917|
|  Chemotherapy|458.39540091909953|
|      Combined| 457.8152186120058|
|       Surgery|457.73744630723684|
+--------------+------------------+



Task 3: Write a function that returns the smoking_status group with the highest survival rate.

In [7]:
from pyspark.sql.functions import round

df.groupBy("smoking_status") \
  .agg((round(avg("survived")*100,2)).alias("survival_rate_percent")) \
  .orderBy(desc("survival_rate_percent")) \
  .show(1)

+--------------+---------------------+
|smoking_status|survival_rate_percent|
+--------------+---------------------+
|  Never Smoked|                22.09|
+--------------+---------------------+
only showing top 1 row


Task 4: Write a function that returns the top three countries with the highest percentage of patients diagnosed in Stage IV. 

In [8]:
from pyspark.sql.functions import count, sum, when, col

country_total = df.groupBy("country") \
                  .agg(count("*").alias("total_patients"))

stage4 = df.withColumn(
    "stage4_flag",
    when(col("cancer_stage") == "Stage IV", 1).otherwise(0)
)

country_stage4 = stage4.groupBy("country") \
                       .agg(sum("stage4_flag").alias("stage4_count"))

result = country_total.join(country_stage4, "country") \
    .withColumn(
        "stage4_percentage",
        (col("stage4_count") / col("total_patients")) * 100
    ) \
    .orderBy(col("stage4_percentage").desc())

result.show(3)

+--------------+--------------+------------+------------------+
|       country|total_patients|stage4_count| stage4_percentage|
+--------------+--------------+------------+------------------+
|        Greece|         33052|        8429| 25.50223889628464|
|       Croatia|         33138|        8426|25.427002233085883|
|Czech Republic|         32885|        8317|25.291166185190818|
+--------------+--------------+------------+------------------+
only showing top 3 rows


Task 5: Write a function that filters patients who:  

Are male  

Diagnosed in Stage III or IV  

Have a family history of cancer  

Are current smokers  

Have a BMI > 30  

Survived 

Return the average age and the percentage of these patients who had hypertension.

In [9]:
from pyspark.sql.functions import avg

filtered = df.filter(
    (col("gender") == "Male") &
    (col("cancer_stage").isin("Stage III", "Stage IV")) &
    (col("family_history") == 1) &
    (col("smoking_status") == "Current Smoker") &
    (col("bmi") > 30) &
    (col("survived") == 1)
)

filtered.select(
    avg("age").alias("average_age"),
    (avg("hypertension") * 100).alias("hypertension_percentage")
).show()

+------------------+-----------------------+
|       average_age|hypertension_percentage|
+------------------+-----------------------+
|55.179398872886665|      74.76518472135254|
+------------------+-----------------------+

